# WDM Time-Frequency Decomposition Of AET

This notebook decomposes the previously generated TDI AET channels into Wilson-Daubechies-Meyer time-frequency maps using `wdm-wavelet`. It uses the noisy AET channels for the main maps and the GW-only AET channels to locate the signal interval.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import decimate
from wdm_wavelet.wdm import WDM

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "lisa_link_sim":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BASE_DIR = PROJECT_ROOT / "lisa_link_sim"
OUTPUT_DIR = BASE_DIR / "outputs"
PLOT_DIR = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

NOISY_AET_CSV = OUTPUT_DIR / "tdi_michelson_XYZ_AET.csv"
GW_AET_CSV = OUTPUT_DIR / "tdi_gw_only_XYZ_AET.csv"
CHANNELS = ["A", "E", "T"]

NOISY_AET_CSV, GW_AET_CSV


(PosixPath('/Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/tdi_michelson_XYZ_AET.csv'),
 PosixPath('/Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/tdi_gw_only_XYZ_AET.csv'))

## Load AET Data

The noisy TDI file stores `A2`, `E2`, and `T2`. The GW-only file stores `A2_gw_hz`, `E2_gw_hz`, and `T2_gw_hz`. Both are already in Hz.


In [2]:
noisy_cols = ["sample", "t_lisa", "A2", "E2", "T2"]
gw_cols = ["sample", "t_lisa", "A2_gw_hz", "E2_gw_hz", "T2_gw_hz"]

noisy_df = pd.read_csv(NOISY_AET_CSV, usecols=noisy_cols)
gw_df = pd.read_csv(GW_AET_CSV, usecols=gw_cols)

if len(noisy_df) != len(gw_df):
    raise ValueError("Noisy and GW-only AET files have different lengths.")
if not np.allclose(noisy_df["t_lisa"].to_numpy(), gw_df["t_lisa"].to_numpy(), rtol=0, atol=1e-6):
    raise ValueError("Noisy and GW-only AET files are not on the same t_lisa grid.")

t_lisa = noisy_df["t_lisa"].to_numpy(dtype=float)
t_rel = t_lisa - t_lisa[0]
dt = float(np.median(np.diff(t_lisa)))

noisy_aet = {
    "A": noisy_df["A2"].to_numpy(dtype=float),
    "E": noisy_df["E2"].to_numpy(dtype=float),
    "T": noisy_df["T2"].to_numpy(dtype=float),
}
gw_aet = {
    "A": gw_df["A2_gw_hz"].to_numpy(dtype=float),
    "E": gw_df["E2_gw_hz"].to_numpy(dtype=float),
    "T": gw_df["T2_gw_hz"].to_numpy(dtype=float),
}

print(f"samples = {len(t_lisa)}")
print(f"dt      = {dt} s")
print(f"span    = {t_rel[-1] - t_rel[0]:.1f} s")
for ch in CHANNELS:
    print(f"{ch}: noisy rms={np.sqrt(np.mean(noisy_aet[ch]**2)):.3e} Hz, gw max={np.max(np.abs(gw_aet[ch])):.3e} Hz")


samples = 1000000
dt      = 0.10000000149011612 s
span    = 99999.9 s
A: noisy rms=1.129e-05 Hz, gw max=3.176e-04 Hz
E: noisy rms=1.097e-05 Hz, gw max=2.955e-04 Hz
T: noisy rms=4.265e-07 Hz, gw max=3.312e-06 Hz


## Locate Signal Interval From GW-Only AET

The WDM maps are shown both with and without the signal overlay. The signal interval is only used for diagnostics and for optional signal/off-source summary statistics.


In [3]:
SIGNAL_THRESHOLD_FRACTION = 1e-3
OFFSOURCE_GUARD_SECONDS = 500.0

gw_power = sum(gw_aet[ch] ** 2 for ch in CHANNELS)
signal_peak_idx = int(np.argmax(gw_power))
signal_t_lisa = float(t_lisa[signal_peak_idx])
signal_peak_rel = signal_t_lisa - t_lisa[0]
threshold = SIGNAL_THRESHOLD_FRACTION * float(np.max(gw_power))
signal_indices = np.flatnonzero(gw_power >= threshold)
signal_window_lisa = (float(t_lisa[signal_indices[0]]), float(t_lisa[signal_indices[-1]]))
signal_window_rel = (signal_window_lisa[0] - t_lisa[0], signal_window_lisa[1] - t_lisa[0])

print(f"signal peak t_lisa = {signal_t_lisa}")
print(f"signal peak rel    = {signal_peak_rel:.1f} s")
print(f"signal window rel  = [{signal_window_rel[0]:.1f}, {signal_window_rel[1]:.1f}] s")
print(f"signal duration    = {signal_window_rel[1] - signal_window_rel[0]:.1f} s")


signal peak t_lisa = 61421184.62766401
signal peak rel    = 49445.2 s
signal window rel  = [45810.2, 51184.0] s
signal duration    = 5373.8 s


## Build WDM Maps

The native AET cadence is `0.1 s`. For this low-frequency LISA band we downsample to `1 Hz` before applying WDM. With `M=1024`, the WDM spacing is about `4.88e-4 Hz` and the TF time pixel is about `1000 s`.


In [4]:
WDM_TARGET_DT = 1.0
WDM_M = 1024
WDM_K = 32
WDM_BETA_ORDER = 6
WDM_PRECISION = 6
WDM_BACKEND = "numba"
WDM_LOW_FREQ_CUT = 1e-5
WDM_HIGH_FREQ_CUT = 1e-2
MIN_NOISE_FLOOR = 1e-300


def downsample_for_wdm(x, dt, target_dt):
    factor = int(round(target_dt / dt))
    if not np.isclose(factor * dt, target_dt):
        raise ValueError("WDM_TARGET_DT must be an integer multiple of the native dt.")
    if factor <= 1:
        return np.asarray(x, dtype=float), dt
    return decimate(np.asarray(x, dtype=float), factor, ftype="fir", zero_phase=True), factor * dt


def wdm_power_map(wdm, x, sample_rate, t0):
    tf = wdm.t2w(x, sample_rate=sample_rate, t0=t0)
    power = np.abs(tf.data) ** 2
    freqs = np.arange(tf.n_freq, dtype=float) * tf.df
    times = tf.t0 + (np.arange(tf.n_time, dtype=float) + 0.5) * tf.dt
    return power, times, freqs, tf


wdm = WDM(M=WDM_M, K=WDM_K, beta_order=WDM_BETA_ORDER, precision=WDM_PRECISION, backend=WDM_BACKEND)
wdm_products = {"noisy": {}, "gw_only": {}}
summary_rows = []

for source_name, source_data in [("noisy", noisy_aet), ("gw_only", gw_aet)]:
    for ch in CHANNELS:
        x_ds, dt_wdm = downsample_for_wdm(source_data[ch], dt, WDM_TARGET_DT)
        power, times_lisa_wdm, freqs_wdm, tf = wdm_power_map(
            wdm, x_ds, sample_rate=1.0 / dt_wdm, t0=t_lisa[0]
        )
        times_rel_wdm = times_lisa_wdm - t_lisa[0]
        signal_tf_mask = (times_lisa_wdm >= signal_window_lisa[0]) & (times_lisa_wdm <= signal_window_lisa[1])
        offsource_tf_mask = (
            (times_lisa_wdm < signal_window_lisa[0] - OFFSOURCE_GUARD_SECONDS)
            | (times_lisa_wdm > signal_window_lisa[1] + OFFSOURCE_GUARD_SECONDS)
        )
        freq_tf_mask = (freqs_wdm >= WDM_LOW_FREQ_CUT) & (freqs_wdm <= WDM_HIGH_FREQ_CUT)

        noise_floor = np.nanmedian(power[:, offsource_tf_mask], axis=1)
        noise_floor = np.where(np.isfinite(noise_floor) & (noise_floor > 0), noise_floor, MIN_NOISE_FLOOR)
        normalized_power = power / noise_floor[:, None]

        signal_band = normalized_power[np.ix_(freq_tf_mask, signal_tf_mask)]
        offsource_band = normalized_power[np.ix_(freq_tf_mask, offsource_tf_mask)]

        wdm_products[source_name][ch] = {
            "power": power,
            "normalized_power": normalized_power,
            "times_lisa": times_lisa_wdm,
            "times_rel": times_rel_wdm,
            "freqs_hz": freqs_wdm,
            "tf": tf,
            "signal_tf_mask": signal_tf_mask,
            "offsource_tf_mask": offsource_tf_mask,
            "freq_tf_mask": freq_tf_mask,
            "noise_floor": noise_floor,
        }

        summary_rows.append({
            "source": source_name,
            "channel": ch,
            "wdm_input_dt_s": dt_wdm,
            "wdm_df_hz": float(tf.df),
            "wdm_dt_tf_s": float(tf.dt),
            "n_time_layers": int(tf.n_time),
            "n_frequency_layers": int(np.sum(freq_tf_mask)),
            "n_signal_time_layers": int(np.sum(signal_tf_mask)),
            "signal_tf_pixels": int(signal_band.size),
            "signal_norm_power_mean": float(np.nanmean(signal_band)),
            "signal_norm_power_median": float(np.nanmedian(signal_band)),
            "signal_norm_power_max": float(np.nanmax(signal_band)),
            "offsource_norm_power_mean": float(np.nanmean(offsource_band)),
            "signal_over_offsource_mean": float(np.nanmean(signal_band) / np.nanmean(offsource_band)),
        })

wdm_summary_df = pd.DataFrame(summary_rows)
WDM_AET_STATS_CSV = OUTPUT_DIR / "aet_wdm_time_frequency_stats.csv"
wdm_summary_df.to_csv(WDM_AET_STATS_CSV, index=False)
print(f"saved {WDM_AET_STATS_CSV}")
wdm_summary_df


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/aet_wdm_time_frequency_stats.csv


,source,channel,wdm_input_dt_s,wdm_df_hz,wdm_dt_tf_s,n_time_layers,n_frequency_layers,n_signal_time_layers,signal_tf_pixels,signal_norm_power_mean,signal_norm_power_median,signal_norm_power_max,offsource_norm_power_mean,signal_over_offsource_mean
0,noisy,A,1.0,0.000488,1020.408178,98,20,5,100,5.243015e+02,36.038992,4.793642e+03,1.565656e+00,3.348765e+02
1,noisy,E,1.0,0.000488,1020.408178,98,20,5,100,2.972305e+02,26.171865,3.131354e+03,1.756831e+00,1.691856e+02
2,noisy,T,1.0,0.000488,1020.408178,98,20,5,100,4.122026e+00,1.101988,4.323381e+01,9.155051e+00,4.502460e-01
3,gw_only,A,1.0,0.000488,1020.408178,98,20,5,100,1.278610e+26,9021.915119,2.842487e+27,5.355700e+19,2.387382e+06
4,gw_only,E,1.0,0.000488,1020.408178,98,20,5,100,8.503694e+25,14935.574310,2.763651e+27,1.020653e+21,8.331620e+04
5,gw_only,T,1.0,0.000488,1020.408178,98,20,5,100,1.256864e+26,14826.018786,2.707516e+27,5.111686e+20,2.458804e+05


## Save WDM Decomposition Arrays

Save raw and off-source-normalized WDM power maps for A, E, and T, for both noisy and GW-only TDI.


In [5]:
save_payload = {
    "signal_window_lisa": np.asarray(signal_window_lisa, dtype=float),
    "signal_window_rel": np.asarray(signal_window_rel, dtype=float),
    "signal_peak_lisa": np.asarray(signal_t_lisa, dtype=float),
    "signal_peak_rel": np.asarray(signal_peak_rel, dtype=float),
}

for source_name in ["noisy", "gw_only"]:
    for ch in CHANNELS:
        prod = wdm_products[source_name][ch]
        prefix = f"{source_name}_{ch}"
        save_payload[f"{prefix}_power"] = prod["power"]
        save_payload[f"{prefix}_normalized_power"] = prod["normalized_power"]
        save_payload[f"{prefix}_noise_floor"] = prod["noise_floor"]
        save_payload[f"{prefix}_times_lisa"] = prod["times_lisa"]
        save_payload[f"{prefix}_times_rel"] = prod["times_rel"]
        save_payload[f"{prefix}_freqs_hz"] = prod["freqs_hz"]

AET_WDM_NPZ = OUTPUT_DIR / "aet_wdm_time_frequency_maps.npz"
np.savez_compressed(AET_WDM_NPZ, **save_payload)
print(f"saved {AET_WDM_NPZ}")


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/aet_wdm_time_frequency_maps.npz


## Plot WDM Maps

Each row is one source (`noisy`, `gw_only`) and each column is one AET channel. The first plot highlights the signal interval; the second plot is the same normalized WDM power without any signal overlay.


In [6]:
def plot_aet_wdm_maps(products, source_name, *, highlight_signal, output_name):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
    last_im = None
    for ax, ch in zip(axes, CHANNELS):
        prod = products[source_name][ch]
        freqs = prod["freqs_hz"]
        times_rel_wdm = prod["times_rel"]
        fmask = (freqs >= 0.0) & (freqs <= WDM_HIGH_FREQ_CUT)
        image = np.log10(prod["normalized_power"][fmask] + 1e-30)
        extent = [
            times_rel_wdm[0] - 0.5 * prod["tf"].dt,
            times_rel_wdm[-1] + 0.5 * prod["tf"].dt,
            freqs[fmask][0],
            freqs[fmask][-1],
        ]
        last_im = ax.imshow(image, aspect="auto", origin="lower", extent=extent, cmap="magma")
        if highlight_signal:
            ax.axvspan(signal_window_rel[0], signal_window_rel[1], color="cyan", alpha=0.20)
            ax.axvline(signal_peak_rel, color="white", ls="--", alpha=0.7)
        ax.set_title(f"{source_name}: {ch}")
        ax.set_xlabel("time in segment [s]")
        ax.grid(alpha=0.15)
    axes[0].set_ylabel("frequency [Hz]")
    fig.colorbar(last_im, ax=axes, label=r"$\log_{10}$ off-source normalized WDM power")
    suffix = "with signal interval" if highlight_signal else "no signal highlight"
    fig.suptitle(f"AET WDM time-frequency maps, {source_name}, {suffix}")
    fig.tight_layout()
    out = PLOT_DIR / output_name
    fig.savefig(out, dpi=150)
    plt.show()
    print(f"saved {out}")

plot_aet_wdm_maps(wdm_products, "noisy", highlight_signal=True, output_name="11_aet_wdm_tf_noisy_highlight.png")
plot_aet_wdm_maps(wdm_products, "noisy", highlight_signal=False, output_name="11_aet_wdm_tf_noisy_no_highlight.png")
plot_aet_wdm_maps(wdm_products, "gw_only", highlight_signal=True, output_name="11_aet_wdm_tf_gw_only_highlight.png")
plot_aet_wdm_maps(wdm_products, "gw_only", highlight_signal=False, output_name="11_aet_wdm_tf_gw_only_no_highlight.png")


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/11_aet_wdm_tf_noisy_highlight.png


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18894/1928782601.py:27: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18894/1928782601.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/11_aet_wdm_tf_noisy_no_highlight.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/11_aet_wdm_tf_gw_only_highlight.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/11_aet_wdm_tf_gw_only_no_highlight.png


## Channel Ranking


In [7]:
for source_name in ["noisy", "gw_only"]:
    display_cols = [
        "source", "channel", "signal_over_offsource_mean", "signal_norm_power_mean",
        "offsource_norm_power_mean", "signal_norm_power_max",
    ]
    print(f"\n{source_name}")
    display(
        wdm_summary_df[wdm_summary_df["source"] == source_name][display_cols]
        .sort_values("signal_over_offsource_mean", ascending=False)
        .reset_index(drop=True)
    )



noisy


,source,channel,signal_over_offsource_mean,signal_norm_power_mean,offsource_norm_power_mean,signal_norm_power_max
0,noisy,A,334.876492,524.301492,1.565656,4793.642000
1,noisy,E,169.185575,297.230483,1.756831,3131.354041
2,noisy,T,0.450246,4.122026,9.155051,43.233812



gw_only


,source,channel,signal_over_offsource_mean,signal_norm_power_mean,offsource_norm_power_mean,signal_norm_power_max
0,gw_only,A,2.387382e+06,1.278610e+26,5.355700e+19,2.842487e+27
1,gw_only,T,2.458804e+05,1.256864e+26,5.111686e+20,2.707516e+27
2,gw_only,E,8.331620e+04,8.503694e+25,1.020653e+21,2.763651e+27
